# Module 1: High Cardinality Aggregations

Welcome! In this module, we will explore a common data engineering workload: high cardinality aggregations.

**Section Goals:**
* Install Faker and generate a mock transactional dataset.
* Run a groupby-sum aggregation on the CPU using Pandas.
* Run the same aggregation on the GPU in parallel using PyTorch.

### Hashing High Cardinality Keys

A "high cardinality" column is one with a large number of unique values (e.g. millions of unique user IDs).

When performing a Groupby on CPU, Pandas must sequentially create a hash map and resolve collisions. This becomes a major bottleneck for large datasets.

On a GPU, we can process millions of keys simultaneously using parallel hashing buckets across CUDA cores, accelerating the aggregation.

### Visualizing Hashing Styles

Here is a whiteboard diagram contrasting sequential hash-mapping on CPU with parallel hash-aggregation on GPU:

![High Cardinality](images/high-cardinality.svg)

### Step 1: Install Faker and Generate Log Data

Let's install Faker and generate a mock dataset containing 100,000 transaction rows with 10,000 unique user IDs.

In [ ]:
!pip install -q faker
from faker import Faker
import random
import pandas as pd
fake = Faker()
data = [{"user_id": fake.random_int(1, 10000), "amount": random.uniform(1.0, 500.0)} for _ in range(100000)]
df = pd.DataFrame(data)

### Step 2: Pandas Groupby on CPU

Let's run a groupby aggregation on the CPU and measure the execution time.

In [ ]:
import time
t0 = time.perf_counter()
res_cpu = df.groupby("user_id")["amount"].sum()
print(f"CPU Groupby Time: {time.perf_counter() - t0:.4f}s")

### Step 3: GPU Parallel Groupby

Now let's copy the data to PyTorch GPU tensors and compute the parallel sum using the GPU's `scatter_add_` function.

In [ ]:
import torch
user_gpu = torch.tensor(df["user_id"].values, device="cuda")
amount_gpu = torch.tensor(df["amount"].values, device="cuda")
torch.cuda.synchronize()
t0 = time.perf_counter()
res_gpu = torch.zeros(10001, device="cuda").scatter_add_(0, user_gpu, amount_gpu)
torch.cuda.synchronize()
print(f"GPU Groupby Time: {time.perf_counter() - t0:.5f}s")

### Interpretation

The parallel GPU calculation is completed significantly faster! PyTorch's `scatter_add_` performs the accumulation on CUDA without needing a sequential Python loop, making it highly optimal for large data pipelines.

### Module 1 Recap

* High cardinality aggregations bottleneck the CPU due to sequential hashing.
* GPUs execute group-by operations in parallel using multi-threaded scatter methods.
* Moving table columns to GPU tensors delivers immediate speedup for data engineering tasks.